<a href="https://colab.research.google.com/github/Thisis4oF/ComfyUIonColab/blob/main/ComfyUIonColab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#`mountUmount(`<font size="3px" color="#01c968">`Gdrive`</font>`)`



In [ ]:
#@markdown <br><center><img src='https://upload.wikimedia.org/wikipedia/commons/thumb/d/da/Google_Drive_logo.png/600px-Google_Drive_logo.png' height="50" alt="Gdrive-logo"/></center>
#@markdown <center><h3>Mount Gdrive to /content/drive</h3></center><br>
MODE = "MOUNT" #@param ["MOUNT", "UNMOUNT"]
#Mount your Gdrive!
from google.colab import drive
drive.mount._DEBUG = False
if MODE == "MOUNT":
  drive.mount('/content/drive', force_remount=True)
elif MODE == "UNMOUNT":
  try:
    drive.flush_and_unmount()
  except ValueError:
    pass
  get_ipython().system_raw("rm -rf /root/.config/Google/DriveFS")

In [33]:
assert __import__('os').path.exists('/content/drive'), 'Google Drive is not mounted yet. Please run the cell above first.'
!ls -la /content/drive

total 28
drwx------  6 root root 4096 Oct 25 14:44 .
drwxr-xr-x  1 root root 4096 Oct 25 14:44 ..
dr-x------  4 root root 4096 Oct 25 14:44 .Encrypted
drwx------ 12 root root 4096 Oct 25 14:44 MyDrive
drwx------  3 root root 4096 Oct 25 14:44 Othercomputers
dr-x------  2 root root 4096 Oct 25 14:44 .shortcut-targets-by-id
drwx------  5 root root 4096 Oct 25 14:44 .Trash-0


#`Setup And Update ComfyUI`



In [34]:
from pathlib import Path

OPTIONS = {}

DRIVE_PATH = "/content/drive/MyDrive/ComfyUI_Drive"  # @param {type:"string"}
UPDATE_COMFY_UI = True  #@param {type:"boolean"}
WORKSPACE = '/content/ComfyUI'
OPTIONS['UPDATE_COMFY_UI'] = UPDATE_COMFY_UI

if DRIVE_PATH:
    WORKSPACE = DRIVE_PATH+"/ComfyUI"
    %cd {DRIVE_PATH}

![ ! -d WORKSPACE ] && echo -= Initial setup ComfyUI =- && git clone https://github.com/comfyanonymous/ComfyUI
%cd $WORKSPACE

if OPTIONS['UPDATE_COMFY_UI']:
  !echo -= Updating ComfyUI =-
  !git pull

!echo -= Install dependencies =-
!pip install xformers!=0.0.18 -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121 --extra-index-url https://download.pytorch.org/whl/cu118 --extra-index-url https://download.pytorch.org/whl/cu117

%cd {WORKSPACE}

/content/drive/MyDrive/ComfyUI_Drive
-= Initial setup ComfyUI =-
fatal: destination path 'ComfyUI' already exists and is not an empty directory.
/content/drive/MyDrive/ComfyUI_Drive/ComfyUI
-= Updating ComfyUI =-
Already up to date.
-= Install dependencies =-
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu121, https://download.pytorch.org/whl/cu118, https://download.pytorch.org/whl/cu117
/content/drive/MyDrive/ComfyUI_Drive/ComfyUI


# `Models Download`

## Download MODELS

In [35]:
#@title 📥 Multi-Category Model Downloader
import os
from google.colab import userdata

# 1. Configuration UI
MODEL_TYPE = "checkpoints" #@param ["checkpoints", "loras", "vae", "controlnet", "upscale_models"]
DOWNLOAD_URL = "https://civitai.com/api/download/models/2071650" #@param {type:"string"}
CUSTOM_FILENAME = "cyberrealisticPony_v127Alt.safetensors" #@param {type:"string"}

# Get Civitai Token if available
try:
  CIVITAI_API_TOKEN = userdata.get('CIVITAI_API_TOKEN')
except:
  CIVITAI_API_TOKEN = ""

# Ensure aria2 is installed
!apt-get -y install -qq aria2

# Set correct destination folder
dest_dir = f"{WORKSPACE}/models/{MODEL_TYPE}"
os.makedirs(dest_dir, exist_ok=True)
print(f"Target directory set to: {dest_dir}")

# Build URL and command
final_url = DOWNLOAD_URL
if 'civitai.com' in DOWNLOAD_URL and CIVITAI_API_TOKEN:
    separator = "&" if "?" in DOWNLOAD_URL else "?"
    final_url = f"{DOWNLOAD_URL}{separator}token={CIVITAI_API_TOKEN}"

out_filename = f"-o \"{CUSTOM_FILENAME}\"" if CUSTOM_FILENAME.strip() else ""

# Execute download
%cd {dest_dir}
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {out_filename} "{final_url}"
%cd {WORKSPACE}
print("Download finished successfully!")

Target directory set to: /content/drive/MyDrive/ComfyUI_Drive/ComfyUI/models/checkpoints
/content/drive/MyDrive/ComfyUI_Drive/ComfyUI/models/checkpoints

10/25 16:35:43 [ERROR] CUID#7 - Download aborted. URI=https://civitai.com/api/download/models/2071650
Exception: [AbstractCommand.cc:351] errorCode=24 URI=https://civitai.com/api/download/models/2071650
  -> [HttpSkipResponseCommand.cc:215] errorCode=24 Authorization failed.

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
c04b1d|ERR |       0B/s|/content/drive/MyDrive/ComfyUI_Drive/ComfyUI/models/checkpoints/cyberrealisticPony_v127Alt.safetensors

Status Legend:
(ERR):error occurred.

aria2 will resume download if the transfer is restarted.
If there are any errors, then see the log file. See '-l' option in help/man page for details.
/content/drive/MyDrive/ComfyUI_Drive/ComfyUI
Download finished successfully!


## OTHERS

In [ ]:
# SD1.5
!wget -c https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.ckpt -P ./models/checkpoints/

# Some SD1.5 anime style
!wget -c https://huggingface.co/WarriorMama777/OrangeMixs/resolve/main/Models/AbyssOrangeMix2/AbyssOrangeMix2_hard.safetensors -P ./models/checkpoints/

# VAE
!wget -c https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors -P ./models/vae/

## LIST MODELS

In [ ]:
!ls -al ./models/checkpoints/

total 6775444
drwxr-xr-x  2 root root       4096 Aug 30 23:40 .
drwxr-xr-x 22 root root       4096 Aug 30 23:39 ..
-rw-r--r--  1 root root 6938040682 Aug 30 23:40 cyberrealisticPony_v127Alt.safetensors
-rw-r--r--  1 root root          0 Aug 30 23:39 put_checkpoints_here


# `START ComfyUI  & Expose Server (MANUAL)`

## Download Prerequisits

In [28]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-10-25 16:22:29--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.10.3/cloudflared-linux-amd64.deb [following]
--2026-10-25 16:22:29--  https://github.com/cloudflare/cloudflared/releases/download/2026.10.3/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/305f141b-9fcc-4a73-94fa-040284dc98f2?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-25T17%3A18%3A41Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b89

In [29]:
!python main.py --dont-print-server

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[INFO] Found comfy_kitchen backend cuda: {'available': True, 'disabled': False, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'closest_point_on_mesh', 'connected_components', 'convrot_w4a4_linear', 'delaunay3d', 'dequantize_convrot_w4a4_weight', 'dequantize_int8_convrot_weight', 'dequantize_int8_convrot_weight_dtype', 'dequantize_int8_simple', 'dequantize_int8_simple_dtype', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'dequantize_w4a8_int8_weight', 'edge_collapse_che

# `START ComfyUI & Expose Server`

## Download Prerequisits

In [30]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-10-25 16:25:06--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.10.3/cloudflared-linux-amd64.deb [following]
--2026-10-25 16:25:06--  https://github.com/cloudflare/cloudflared/releases/download/2026.10.3/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/305f141b-9fcc-4a73-94fa-040284dc98f2?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-10-25T17%3A11%3A14Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b89

## CF Tunnel

In [36]:
import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch cloudflared (if it gets stuck here cloudflared is having issues)\n")

  p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:{}".format(port)], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
  for line in p.stderr:
    l = line.decode()
    if "trycloudflare.com " in l:
      print("This is the URL to access ComfyUI:", l[l.find("http"):], end='')
    #print(l, end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

!python main.py --dont-print-server

[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[INFO] Found comfy_kitchen backend hip: {'available': False, 'disabled': False, 'unavailable_reason': 'PyTorch ROCm/HIP runtime not available', 'capabilities': []}
[INFO] Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_rope1_', 'apply_rope_', 'apply_rope_split_half', 'apply_rope_split_half1', 'apply_rope_split_half1_', 'apply_rope_split_half_', 'dequantize_nvfp4', 'dequantize_per_tensor_fp8', 'int8_linear', 'na3d', 'quantize_and_rotate_rowwise', 'quantize_int8_rowwise', 'quantize_mxfp8', 'quantize_nvfp4', 'quantize_per_tensor_

## localtunnel

In [32]:
# localtunnel
!npm install -g localtunnel

import subprocess
import threading
import time
import socket
import urllib.request

def iframe_thread(port):
  while True:
      time.sleep(0.5)
      sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
      result = sock.connect_ex(('127.0.0.1', port))
      if result == 0:
        break
      sock.close()
  print("\nComfyUI finished loading, trying to launch localtunnel (if it gets stuck here localtunnel is having issues)\n")

  print("The password/enpoint ip for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
  p = subprocess.Popen(["lt", "--port", "{}".format(port)], stdout=subprocess.PIPE)
  for line in p.stdout:
    print(line.decode(), end='')


threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()
%cd /content/ComfyUI
!python main.py --dont-print-server

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
changed 22 packages in 1s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼/content/ComfyUI
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[INFO] Found comfy_kitchen backend hip: {'available': False, 'disabled': False, 'unavailable_reason': 'PyTorch ROCm/HIP runtime not available', 'capabilities': []}
[INFO] Found comfy_kitchen backend ascend: {'available': False, 'disabled': False, 'unavailable_reason': "torch-npu is not installed: No module named 'torch_npu'", 'capabilities': []}
[INFO] Found comfy_kitchen backend triton: {'available': True, 'disabled': True, 'unavailable_reason': None, 'capabilities': ['adaln', 'apply_rope', 'apply_rope1', 'apply_ro